# Conversation History and Basic Memory

**Course author:** Jayneel Girnara

A chat model call is stateless unless prior messages are supplied. This lesson demonstrates that behavior, then introduces manually managed conversation history.

## Learning goals
- Observe what a model does and does not remember between independent calls.
- Represent a conversation as an ordered list of role/content messages.
- Maintain history during a simple interactive chat loop.

## Before you start
Install dependencies and configure `OPENAI_API_KEY` in your local .env file. Do not commit credentials.

## How to work through it
Run the cells top to bottom. Compare the independent requests first, then inspect how adding earlier turns to the message list changes the interaction. Stop the input loop with `quit`, `exit`, or `bye`.

## 1. Observe stateless model calls

Each `.invoke` sends only the prompt passed in that call. The model does not automatically retain your name from a previous independent request. If the second call does not include the earlier message, the model has no conversation history from that call.

**Checkpoint:** Why can’t the second independent request reliably answer “What is my name?”?

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [ ]:
res=llm.invoke("Hello,my name is Alex Example")
res.content

In [ ]:
res=llm.invoke("What is my name")
res.content

In [ ]:
prompts=[
    {"role":'user',"content":"Hello my name is Alex Example"},
    {"role":"ai","content":"Hello, Alex! How can I assist you today?"},
    {"role":"user","content":"What is my name"}

]

res=llm.invoke(prompts)
res.content

## 2. Manage conversation history

Passing earlier turns back to the model provides conversational context. The loop below stores turns explicitly; this is separate from durable database-backed memory.

A **conversation history** is the ordered list of messages the app sends on each turn. This loop appends the user message, calls the model with all saved turns, then appends the assistant response. This is in-process Python state: it is lost when the kernel stops and grows the prompt on every turn. Persistent memory needs storage and careful limits.

**Checkpoint:** Which messages are sent to the model on the third turn?

In [ ]:
history=[]
while True:
    query=input("user: ")
    if query.lower() in ["exit","quit","bye"]:
        print("Good Bye")
        break

    history.append({"role":"user","content":query})
    print("User:",query)

    res=llm.invoke(history)
    history.append({"role":"ai","content":res.content})

    print("Ai",res.content)